# 01 — Prepare Symile exports and audit patient splits

Run all cells in order after notebook 00. This notebook creates destination directories, then either loads existing exports or **assembles them from prepared source files**. It does not run PULSE/MedGemma, encode sensors, or invent independent labels.

In real mode, missing source files produce an actionable **NOT READY** message and no empty `records.json`/`targets.json`. Do not run notebook 02 until the last cell says **READY**. Synthetic mode remains available for testing and never substitutes for missing real data.

If starting from raw Symile only, extract the source tables described below first. Native linkage, study selection, acquisition times, sensor checksums and independent target provenance must be verified upstream.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
assert (ROOT / 'core.py').is_file(), 'Start the kernel in graph_reasoning/'
sys.path.insert(0, str(ROOT))
from core import audit, fixture, load, save

CONFIG = load(ROOT / 'config.json')
if CONFIG.get('mode') not in ('synthetic', 'real'):
    raise ValueError("config.json mode must be 'synthetic' or 'real'")

def resolve_path(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError('Expected a nonempty path string')
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path).resolve()

OUT = resolve_path(CONFIG['output_dir'])
OUT.mkdir(parents=True, exist_ok=True)
READY = False
rows = targets = audit_result = None  # Prevent reuse of stale notebook variables.
EXPORT_PATHS = {}
if CONFIG['mode'] == 'real':
    for key in ('records_path', 'targets_path'):
        EXPORT_PATHS[key] = resolve_path(CONFIG.get(key))
        EXPORT_PATHS[key].parent.mkdir(parents=True, exist_ok=True)
    if len(set(EXPORT_PATHS.values())) != 2:
        raise ValueError('records_path and targets_path must be different files')
    if set(EXPORT_PATHS.values()) & {OUT / 'records.json', OUT / 'targets.json', OUT / 'audit.json'}:
        raise ValueError('Keep input exports separate from output_dir artifacts')
    for key, path in EXPORT_PATHS.items():
        print(f'{key}: {path} | exists: {path.is_file()}')
print('Mode:', CONFIG['mode'])
print('Output directory:', OUT)


## Source files for creating real exports

If both configured exports already exist, leave this cell unchanged. Otherwise, set the following four paths to **existing, prepared JSON arrays**. All tables join on the same `id`; every selected record must occur exactly once in each. You can alternatively store these paths under `export_sources` in `config.json`. Paths below default to `None`, not nonexistent sample files.

| Source | Required fields for each entry |
|---|---|
| `manifest` | `id`, `patient_id`, `admission_id`, `split` (`train`/`val`/`test`), `modality` (`ecg`/`cxr`) |
| `embeddings` | `id`, `embedding` (numeric vector), `embedding_space` (encoder/preprocessing version) |
| `candidates` | `id`, `candidates` (list of sensor-derived findings) |
| `annotations` | `id`, `findings` (concept → 1/0/null), `target_source`, `target_revision` |

Each candidate requires `concept`, `polarity`, `model`, `revision`, `sensor_sha256`, `text`, and `quote`; see `DATA_CONTRACT.md`. Model identifiers are `PULSE-ECG/PULSE-7B` for ECG and `google/medgemma-1.5-4b-it` for CXR. Combine modality exports into the source arrays first. This cell cannot automatically map an unknown raw dataset format.

**Annotations must be independent of the candidate outputs.** Missing/uncertain labels remain null or omitted; they are not converted to negatives. Supply a complete, prespecified patient split in the manifest. Admission pairing alone is not a finding label.

In [ ]:
# EDIT these paths if records.json and targets.json have not been created.
# Each path must point to an existing JSON array following the table above.
SOURCE_PATHS = {
    'manifest': None,
    'embeddings': None,
    'candidates': None,
    'annotations': None,
}
# Optional persistent configuration; these entries override the values above.
SOURCE_PATHS.update(CONFIG.get('export_sources', {}))

for name in ('manifest', 'embeddings', 'candidates', 'annotations'):
    value = SOURCE_PATHS.get(name)
    print(name, ':', value if value else 'not configured')


## Validate source tables and assemble exports

The next cell defines the assembler. It rejects duplicate/missing IDs, invalid arrays, synthetic markers, and incomplete schemas before saving either export. The standard patient/modality audit then checks the assembled cohort. It only accepts explicit finding states; no labels or embeddings are generated here. Provenance strings cannot independently prove annotation quality—review the underlying sources.

In [ ]:
def indexed_source(path, required, name):
    entries = load(path)
    if not isinstance(entries, list) or not entries:
        raise ValueError(f'{name}: expected a nonempty JSON array')
    result = {}
    for number, entry in enumerate(entries):
        if not isinstance(entry, dict):
            raise ValueError(f'{name}[{number}]: expected an object')
        missing = set(required) - entry.keys()
        if missing:
            raise ValueError(f'{name}[{number}]: missing fields {sorted(missing)}')
        rid = entry['id']
        if not isinstance(rid, str) or not rid.strip() or rid in result:
            raise ValueError(f'{name}: invalid or duplicate record id at row {number}')
        if entry.get('synthetic') not in (None, False):
            raise ValueError(f'{name}: synthetic input is not allowed in real exports')
        result[rid] = entry
    return result


def assemble_exports(source_paths):
    schemas = {
        'manifest': ('id', 'patient_id', 'admission_id', 'split', 'modality'),
        'embeddings': ('id', 'embedding', 'embedding_space'),
        'candidates': ('id', 'candidates'),
        'annotations': ('id', 'findings', 'target_source', 'target_revision'),
    }
    tables = {name: indexed_source(resolve_path(source_paths[name]), fields, name)
              for name, fields in schemas.items()}
    expected = set(tables['manifest'])
    for name, table in tables.items():
        if set(table) != expected:
            raise ValueError(f'{name}: ID coverage differs from manifest '
                             f'(missing={len(expected-set(table))}, extra={len(set(table)-expected)})')
    assembled_rows, assembled_targets = [], []
    for rid, manifest in tables['manifest'].items():
        embedding = tables['embeddings'][rid]
        candidate_list = tables['candidates'][rid]['candidates']
        annotation = tables['annotations'][rid]
        for field in ('patient_id', 'admission_id', 'split', 'modality'):
            if not isinstance(manifest[field], str) or not manifest[field].strip():
                raise ValueError(f'{rid}: {field} must be a nonempty string')
        if not isinstance(candidate_list, list) or any(not isinstance(c, dict) for c in candidate_list):
            raise ValueError(f'{rid}: candidates must be a list of objects')
        if not isinstance(annotation['findings'], dict):
            raise ValueError(f'{rid}: findings must be an object mapping concepts to 1/0/null')
        if not isinstance(embedding['embedding'], list) or not embedding['embedding']:
            raise ValueError(f'{rid}: embedding must be a nonempty numeric list')
        if not isinstance(embedding['embedding_space'], str) or not embedding['embedding_space'].strip():
            raise ValueError(f'{rid}: embedding_space must be a nonempty string')
        if annotation['target_source'] in ('synthetic_fixture', 'PULSE-ECG/PULSE-7B', 'google/medgemma-1.5-4b-it'):
            raise ValueError(f'{rid}: supply independent target annotations, not candidate/model outputs')
        for candidate in candidate_list:
            for field in ('concept', 'polarity', 'model', 'revision', 'sensor_sha256', 'text', 'quote'):
                if not isinstance(candidate.get(field), str) or not candidate[field].strip():
                    raise ValueError(f'{rid}: candidate {field} must be a nonempty string')
            checksum = candidate['sensor_sha256']
            if len(checksum) != 64 or any(c not in '0123456789abcdefABCDEF' for c in checksum):
                raise ValueError(f'{rid}: invalid sensor SHA-256')
        # Whitelist record fields: no target annotations enter model inputs.
        assembled_rows.append({**{k: manifest[k] for k in schemas['manifest']},
            'embedding': embedding['embedding'], 'embedding_space': embedding['embedding_space'],
            'candidates': [{k: c[k] for k in ('concept', 'polarity', 'model', 'revision',
                           'sensor_sha256', 'text', 'quote')} for c in candidate_list],
            'synthetic': False})
        assembled_targets.append({k: annotation[k] for k in schemas['annotations']})
    result = audit(assembled_rows, assembled_targets, synthetic=False)
    return assembled_rows, assembled_targets, result


def write_new_exports(prepared_rows, prepared_targets):
    # Serialize both first; never create placeholders or overwrite an existing export.
    payloads = [json.dumps(value, indent=2, allow_nan=False) for value in (prepared_rows, prepared_targets)]
    paths = [EXPORT_PATHS['records_path'], EXPORT_PATHS['targets_path']]
    if any(p.exists() for p in paths):
        raise FileExistsError('An export already exists; use both existing exports or choose two new paths')
    created = []
    try:
        for path, payload in zip(paths, payloads):
            with path.open('x', encoding='utf-8') as handle:
                created.append(path)
                handle.write(payload)
    except Exception:
        for path in created:
            path.unlink(missing_ok=True)
        raise


## Load or create, audit, then save the notebook inputs

Run this cell after configuring the source paths. If inputs are missing, it prints what to supply and leaves the notebook **NOT READY**. After supplying files, rerun from the configuration cell. It never falls back to synthetic data in real mode. Existing audited output files are left untouched while preparation is incomplete; do not use them as evidence that the current run is ready.

In [ ]:
READY = False
rows = targets = audit_result = None
if CONFIG['mode'] == 'synthetic':
    rows, targets = fixture()
else:
    existence = {name: path.is_file() for name, path in EXPORT_PATHS.items()}
    if all(existence.values()):
        rows = load(EXPORT_PATHS['records_path'])
        targets = load(EXPORT_PATHS['targets_path'])
        print('Loaded both existing real exports.')
    elif any(path.exists() for path in EXPORT_PATHS.values()):
        print('NOT READY: only one export exists, or an export path is a directory.')
        print('Supply a matching pair of files or choose two new export filenames; existing data were not overwritten.')
    else:
        missing_sources = []
        for name in ('manifest', 'embeddings', 'candidates', 'annotations'):
            value = SOURCE_PATHS.get(name)
            if not value:
                missing_sources.append(f'{name}: path not configured')
            elif not resolve_path(value).is_file():
                missing_sources.append(f'{name}: file not found: {resolve_path(value)}')
        if missing_sources:
            print('NOT READY: export directories exist, but real source data are still required.')
            print('\n'.join(missing_sources))
            print('Set SOURCE_PATHS above. Raw Symile alone still needs sensor embeddings, PULSE/MedGemma extraction, and independent annotations.')
            print('No empty or synthetic real-data exports were created. Do not run notebook 02 yet.')
        else:
            rows, targets, audit_result = assemble_exports(SOURCE_PATHS)
            write_new_exports(rows, targets)
            print('Created validated real exports:')
            print(*EXPORT_PATHS.values(), sep='\n')

if rows is not None and targets is not None:
    if not isinstance(rows, list) or not isinstance(targets, list):
        raise ValueError('records.json and targets.json must each contain a JSON array')
    audit_result = audit(rows, targets, CONFIG['mode'] == 'synthetic')
    save(OUT / 'records.json', rows)
    save(OUT / 'targets.json', targets)
    save(OUT / 'audit.json', audit_result)
    READY = True
    print('READY: exports passed the patient/modality audit. Continue to notebook 02.')
    print(audit_result)
